# QAOA for MaxCut

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/tutorials/qaoa-maxcut.ipynb)

Split a network to cut the most connections. The shape of real routing, scheduling, and logistics problems.

Nothing in this notebook costs anything or needs an account until the last section. The result shown is read from the public certificate for the run that actually produced it.

Full write-up: [QAOA for MaxCut](https://zksf.org/blog/qaoa-maxcut-optimization-tutorial/)


## The idea

MaxCut asks how to divide the nodes of a graph into two groups so that the largest number of edges runs between the groups rather than inside them.

QAOA alternates two layers. A **cost** layer applies a phase proportional to how good each candidate answer is, and a **mixer** layer lets amplitude flow between candidates. Repeating them concentrates probability on the good cuts.

The graph here is a four node ring, 0-1-2-3-0. The best answer is the alternating split, so the circuit should favour `1010` and `0101`.


## The circuit


In [ ]:
!pip install -q qiskit


In [ ]:
from qiskit import QuantumCircuit

gamma, beta = 2.3, 0.86      # angles, tuned in advance for p=1
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]

qc = QuantumCircuit(4, 4)
qc.h(range(4))               # every partition equally likely

for a, b in edges:           # cost layer: phase per edge
    qc.cx(a, b)
    qc.rz(gamma, b)
    qc.cx(a, b)

qc.rx(beta, range(4))        # mixer layer

qc.measure(range(4), range(4))
print(qc.draw(output='text'))


## What happened when this ran

The rz rotations are non-Clifford, so the stabilizer engine cannot take this one. At four qubits the router chose exact statevector, which is cheap and carries no approximation error.

The cell below reads the real certificate for that run. No account, no cost.


In [ ]:
import requests

CERT = "bdef4e3e31f940d4"
c = requests.get(f"https://api.zksf.org/certify/{CERT}/json", timeout=30).json()

print(f"engine   : {c['engine']}")
print(f"method   : {c['method']}")
print(f"shots    : {c['shots']}")
if c.get('expectation') is not None:
    print(f"energy   : {c['expectation']:.6f}")
for bits, n in (c.get('top_outcomes') or []):
    print(f"  {bits}  {n}")
print(f"accuracy : {c.get('error_bound', 'exact, shot noise only')}")
print(f"verify   : {c['verify_url']}")


## Reading the result

Read the tiers. The two optimal alternating partitions, `1010` and `0101`, together took about 56 percent of the shots. The next-best cuts split most of the rest. The two worst answers, `0000` and `1111`, which cut nothing at all, drew under 5 percent between them. A single shallow layer has already pushed the measurement onto the good cuts.

That certificate is public. Anyone can open the verify link, or check it programmatically without an account:

```
pip install zcc-verify
zcc-verify bdef4e3e31f940d4
```


## Run it yourself

Optional, and this part does cost. Circuits this small are a fraction of a cent, and `estimate()` prices any job for free before you commit to it. Get a token from [app.zksf.org](https://app.zksf.org).


In [ ]:
!pip install -q qsim-sdk


In [ ]:
import getpass
import qsim_sdk

client = qsim_sdk.Client(token=getpass.getpass("ZKSF API token: "))

est = client.estimate(qc, shots=1000)
print('engine:', est['engine'], '| cost: $', est['predicted_cost_usd'])


In [ ]:
job = client.run(qc, shots=1000)

print(job['result'].get('counts'))
print(job['result'].get('expectation'))
print(job['result']['error_info'])


## Next

- [The full article](https://zksf.org/blog/qaoa-maxcut-optimization-tutorial/), with the maths and the background
- [All tutorials](https://zksf.org/blog/) and the [glossary](https://zksf.org/glossary-of-essential-quantum-computing-terms-for-beginners/)
- [How the accuracy statements work](https://zksf.org/quantum-computing-certification/), and [the paper](https://doi.org/10.5281/zenodo.21851381)
- [Quickstart notebook](https://colab.research.google.com/github/official-dvl/zksf/blob/main/examples/quickstart.ipynb): four certified runs, including one on real quantum hardware
